# SW05 / KW42 - Datenbankzugriff via Python + Pandas + Arrays/Listen

## Einfuehrung
In dieser Woche verbindet ihr Listen, Pandas und SQL zu einer durchgaengigen Datenpipeline.
Ihr bereitet Daten in Python auf, speichert sie in einer Datenbank und wertet sie wieder aus.
Am Ende koennt ihr tabellarische Daten zwischen Python, Pandas und SQL sicher ueberfuehren.

## Lernziele dieser Woche
- Listen strukturierter Daten fuer tabellarische Verarbeitung vorbereiten.
- DataFrames erstellen und zentrale Pandas-Operationen anwenden.
- Mit sqlite3 Daten schreiben, lesen und mit SQL filtern.
- Abfrageergebnisse zur Weiterverarbeitung in Pandas zurueckfuehren.

## Rueckblick / Baut auf
- Baut explizit auf Woche 2 auf (Datentypen, Variablen, Funktionen).
- Baut auf Woche 3 auf (Bedingungen) und Woche 4 (Schleifen) fuer Datenbereinigung und Iteration.

## Ablauf der Lektion
1. Listen von Dictionaries aufbauen.
2. Daten in DataFrames ueberfuehren.
3. Daten in SQL speichern und abfragen.
4. Resultate vergleichen und visualisieren.


# SW05 / KW42 - Datenbankzugriff via Python + Pandas + Arrays/Listen

In dieser Unterrichtseinheit verbinden wir drei zentrale Themen:
1. Listen als grundlegende Python-Datenstruktur
2. Pandas als Werkzeug fuer tabellarische Daten
3. Datenbankzugriff via Python mit `sqlite3`

Ziel ist ein durchgaengiger Datenfluss: **Liste -> DataFrame -> Datenbank -> Query-Resultat -> DataFrame**.

## Was ihr schon koennt (Rueckblick aus den letzten Wochen)
- **Woche 2:** Datentypen, Variablen, Funktionen (`print`, `type`, eigene Werte speichern)
- **Woche 3:** `if`/`else`, boolesche Bedingungen (`==`, `>=`, `and`, `or`)
- **Woche 4:** Schleifen (`for`, `while`) fuer wiederholte Verarbeitung

Heute nutzen wir genau diese Grundlagen weiter - nur auf **tabellarischen Daten**.

## Lernziele
- Listen (Arrays) wiederholen und strukturierte Daten vorbereiten.
- Mit Pandas DataFrames erstellen und wichtige Grundoperationen anwenden.
- Mit `sqlite3` eine Datenbank aufbauen, Daten schreiben und gezielt abfragen.
- Verstehen, wie Datenformate zwischen Python, Pandas und SQL wechseln.

## Fahrplan der Lektion
1. Listen von Dictionaries aufbauen (reines Python)
2. Diese Liste in einen Pandas DataFrame umwandeln
3. Dieselben Daten in SQL speichern und abfragen
4. Query-Resultate wieder in Pandas analysieren

In [ ]:
import sqlite3
import pandas as pd

## 1) Listen als Ausgangspunkt

Listen (`list`) sind in Python geordnete Sammlungen von Elementen.
Fuer tabellarische Daten ist eine **Liste von Dictionaries** besonders praktisch:
- jedes Dictionary entspricht einer Zeile,
- jeder Key entspricht spaeter einer Spalte.

In [ ]:
students = [
    {"id": 1, "name": "Anna", "punkte": 84},
    {"id": 2, "name": "Ben", "punkte": 71},
    {"id": 3, "name": "Clara", "punkte": 93},
    {"id": 4, "name": "Dario", "punkte": 65},
]

students

### Verbindung zu Woche 2 und 4
- Wie bei allen Datentypen kann man mit `type(...)` prüfen, was vorliegt.
- Wie bei Strings funktioniert Indexzugriff mit `[0]`, `[1]`, ...
- Mit `for`-Schleifen koennen wir jeden Datensatz einzeln verarbeiten.

In [ ]:
print(type(students))
print(type(students[0]))
print("Anzahl Datensaetze:", len(students))
print("Erster Name:", students[0]["name"])

In [ ]:
for eintrag in students:
    print(eintrag["name"], "hat", eintrag["punkte"], "Punkte")

## 2) Von Liste zu Pandas DataFrame

Pandas eignet sich fuer tabellarische Daten. Ein `DataFrame` ist eine Tabelle mit Zeilen und Spalten.

Gedanklich ist ein DataFrame sehr nah an einer Excel-Tabelle:
- Spaltennamen oben (`id`, `name`, `punkte`)
- Pro Zeile ein Datensatz
- Numerische Spalten koennen direkt berechnet/gefiltert werden

In [ ]:
df_students = pd.DataFrame(students)
df_students

### DataFrame lesen koennen
- `shape` zeigt (Zeilen, Spalten)
- `columns` zeigt die Spaltennamen
- `dtypes` zeigt die Datentypen je Spalte

Das ist spaeter wichtig, wenn wir SQL-Tabellen korrekt definieren.

In [ ]:
print("Shape:", df_students.shape)
print("Columns:", list(df_students.columns))
df_students.dtypes

### Verbindung zu Woche 3 (Bedingungen)
Filter in Pandas funktionieren wie `if`-Bedingungen - nur fuer viele Zeilen auf einmal.

In [ ]:
durchschnitt = df_students["punkte"].mean()
bestanden = df_students[df_students["punkte"] >= 70]

print(f"Durchschnitt: {durchschnitt:.2f}")
bestanden

### Warum funktioniert der Filter?
Der Ausdruck `df_students["punkte"] >= 70` erzeugt pro Zeile einen `True`/`False`-Wert.
Diese Bool-Werte werden als Maske verwendet, um nur passende Zeilen auszuwaehlen.

In [ ]:
maske = df_students["punkte"] >= 70
maske

Wir koennen dasselbe auch mit einer Schleife (Woche 4) formulieren.
Das Resultat ist gleich, Pandas ist aber kuerzer und schneller auf grossen Datenmengen.

In [ ]:
bestanden_namen = []
for s in students:
    if s["punkte"] >= 70:
        bestanden_namen.append(s["name"])

bestanden_namen

## 3) DB-Zugriff via Python (`sqlite3`)

`sqlite3` ist Teil der Python-Standardbibliothek. Wir nutzen eine In-Memory-Datenbank (`:memory:`):
- schnell fuer Uebungen,
- keine Datei noetig,
- nach Programmende wieder weg.

Wichtige Rollen:
- `con` (Connection): Verbindung zur Datenbank
- `cur` (Cursor): fuehrt SQL-Befehle aus

### SQL-Begriffe kurz erklaert
- `CREATE TABLE`: erstellt eine neue Tabelle
- `INSERT`: schreibt neue Zeilen hinein
- `SELECT`: liest Daten wieder aus
- `WHERE`: filtert Zeilen (wie eine Bedingung)
- `ORDER BY`: sortiert Resultate

In [ ]:
con = sqlite3.connect(":memory:")
cur = con.cursor()

cur.execute("""
CREATE TABLE studenten (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    punkte INTEGER NOT NULL
)
""")

rows = [(s["id"], s["name"], s["punkte"]) for s in students]
cur.executemany("INSERT INTO studenten (id, name, punkte) VALUES (?, ?, ?)", rows)
con.commit()

cur.execute("SELECT id, name, punkte FROM studenten WHERE punkte >= ? ORDER BY punkte DESC", (70,))
result = cur.fetchall()
result

### Hinweis zu SQL-Parametern
Wir verwenden Platzhalter `?` statt String-Verkettung.
Das ist robuster, besser lesbar und reduziert Fehler bei Datentypen.

### Kleine Kontrolle
Wie viele Zeilen liegen in der SQL-Tabelle?

In [ ]:
cur.execute("SELECT COUNT(*) FROM studenten")
cur.fetchone()

## 4) Query-Resultat wieder als DataFrame nutzen

SQL liefert mit `fetchall()` eine Liste von Tupeln.
Fuer weitere Analysen ist ein DataFrame oft bequemer.

In [ ]:
df_result = pd.DataFrame(result, columns=["id", "name", "punkte"])
df_result

Jetzt koennen wir sofort wieder Pandas-Operationen nutzen.

In [ ]:
print("Anzahl bestandene Studierende:", len(df_result))
print("Bester Wert:", df_result["punkte"].max())

## 5) Typische Stolpersteine (und wie ihr sie erkennt)
- `KeyError`: Spaltenname oder Dictionary-Key falsch geschrieben
- `NameError`: Variable wurde noch nicht erstellt oder heisst anders
- SQL-Syntaxfehler: oft Tippfehler bei `SELECT`, `WHERE`, `ORDER BY`

Vorgehen bei Fehlern:
1. Fehlermeldung von oben nach unten lesen
2. Zeile in der Meldung suchen
3. Datentypen/Spaltennamen kontrollieren (`type`, `dtypes`, `columns`)

## 6) Zusammenfassung
- Listen eignen sich als einfache Datenquelle im Python-Code.
- Pandas hilft beim Strukturieren und Auswerten.
- Mit `sqlite3` koennen wir Daten speichern und mit SQL filtern/sortieren.
- Der Wechsel zwischen Python-Objekten, DataFrames und SQL-Resultaten ist eine Kernkompetenz fuer datenorientierte Anwendungen.

Wenn ihr diesen Ablauf versteht, koennt ihr bereits einfache Datenanwendungen Ende-zu-Ende bauen.

In [ ]:
con.close()